In [1]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedKFold,
    cross_val_predict,
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
)

# 폰트 및 표시 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
pd.options.display.float_format = '{:.2f}'.format

# 데이터 로드
df = pd.read_csv('dataset/Telco-Customer-Churn.csv')
print('데이터 크기:', df.shape)
df.head()
df.info()

# 컬럼 의미
# customerID        : 고객 고유번호 (모델 입력에서 제외)
# gender            : 성별
# SeniorCitizen     : 고령 여부 (0/1)
# Partner           : 배우자 여부
# Dependents        : 부양가족 여부
# tenure            : 이용 개월 수
# PhoneService      : 전화 서비스
# MultipleLines     : 다중회선
# InternetService   : 인터넷 종류 (DSL / Fiber optic / No)
# OnlineSecurity 등 : 부가 서비스
# Contract          : 계약 유형 (월단위 / 1년 / 2년)
# PaperlessBilling  : 전자청구서
# PaymentMethod     : 결제 방식
# MonthlyCharges    : 월 요금
# TotalCharges      : 총 요금
# Churn             : 이탈 여부 (Yes/No)  ← 예측 대상

# 결측치 확인
print(df.isnull().sum())

# TotalCharges는 숫자처럼 보이지만 Object 타입이다.
# tenure=0인 신규 고객은 빈 문자열(' ')이라 숫자 변환이 필요하다.
print('\nTotalCharges 타입:', df['TotalCharges'].dtype)
print('빈 문자열 개수:', (df['TotalCharges'].astype(str).str.strip() == '').sum())

# 숫자로 바꾸고, 변환 실패(빈 값)는 NaN → 0으로 채움
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print('변환 후 결측 수:', df['TotalCharges'].isnull().sum())
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# 식별자는 예측에 필요 없으므로 제거
df = df.drop('customerID', axis=1)

# 타깃을 숫자로 변환 (Yes=1 이탈, No=0 유지)
df['Churn'] = df['Churn'].map({'No': 0, 'Yes': 1})

print('\n전처리 후 크기:', df.shape)
print(df[['tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']].head())


# 이탈 비율 확인
print(df['Churn'].value_counts())
print('\n이탈 비율:')
print(df['Churn'].value_counts(normalize=True))

plt.figure(figsize=(5, 4))
sns.countplot(x='Churn', data=df)
plt.xticks([0, 1], ['유지 (0)', '이탈 (1)'])
plt.title('고객 이탈 여부 분포')
plt.show()

# 계약 유형 / 이용 기간과 이탈의 관계
###fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

###sns.countplot(x='Contract', hue='Churn', data=df, ax=axes[0])
##sns.countplot(x='Contract', hue='Churn', data=df, ax=axes[0])

plt.figure(figsize=(5, 4))##테스트 추가
sns.countplot(x='Contract', hue='Churn', data=df)##테스트
plt.title('계약 유형별 이탈')
plt.legend(title='유지', labels=['유지', '이탈'])
plt.show()

plt.figure(figsize=(5, 4))

sns.boxplot(x='Churn', y='tenure', data=df)
plt.xticks([0, 1],['유지 (0)', '이탈 (1)'])
plt.title('이용 기간(tenure)과 이탈')
plt.show()

# 수치형 변수 상관관계
num_cols = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']
plt.figure(figsize=(7, 5))
sns.heatmap(df[num_cols].corr(), annot=True, cmap='coolwarm')
plt.title('수치형 변수 상관관계')
plt.show()



# 2. 입력 변수(X)와 타겟 변수(y) 설정
X = df.drop('Churn', axis=1)
y = df['Churn']

# 기존 모델과 공정하게 비교할 수 있도록 변환 전 열 순서를 보존한다.
original_numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

# 2-1. 이진화 가능한 범주형 변수 자동 탐지 및 0/1 변환
# 고유값이 정확히 2개인 문자열 컬럼은 원핫인코딩 대신 한 개의 0/1 열로 직접 변환한다.
# No/Yes는 No=0, Yes=1로 고정하고, 그 외 이진 범주는 정렬된 첫 값을 0, 둘째 값을 1로 둔다.
string_cols = X.select_dtypes(include=['object', 'str']).columns.tolist()
binary_cols = [col for col in string_cols if X[col].nunique(dropna=False) == 2]
binary_mappings = {}

for col in binary_cols:
    values = X[col].dropna().unique().tolist()
    if set(values) == {'No', 'Yes'}:
        mapping = {'No': 0, 'Yes': 1}
    else:
        sorted_values = sorted(values)
        mapping = {sorted_values[0]: 0, sorted_values[1]: 1}
    binary_mappings[col] = mapping
    X[col] = X[col].map(mapping).astype('int64')

# SeniorCitizen은 원본부터 0/1 수치형이고, Churn은 위에서 No=0/Yes=1로 변환했다.
print('자동 이진화 대상 및 매핑:')
for col, mapping in binary_mappings.items():
    print(f'  {col}: {mapping}')
print("기존 이진 수치형: SeniorCitizen {'해당 없음': 0, '고령': 1}")
print("타깃 이진화: Churn {'No': 0, 'Yes': 1}")

# 3개 이상 범주만 원핫인코딩하고, 이진화된 열은 수치형 입력으로 사용한다.
categorical_cols = X.select_dtypes(include=['object', 'str']).columns.tolist()
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print('수치형(이진화 열 포함):', numeric_cols)
print('원핫인코딩 대상:', categorical_cols)

# 데이터 분할 (Train 80%, Test 20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('\n학습 데이터:', X_train.shape)
print('테스트 데이터:', X_test.shape)


# OneHotEncoder: 이진화 후 남은 3개 이상 범주형에만 적용
# - sparse_output=False : 배열로 받음
# - handle_unknown='ignore' : 학습에 없던 값은 0으로
# - drop='first' : 더미 변수 함정 방지
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore', drop='first')

X_train_cat = encoder.fit_transform(X_train[categorical_cols])
X_test_cat = encoder.transform(X_test[categorical_cols])

encoded_cat_cols = encoder.get_feature_names_out(categorical_cols)

# 수치형 + 인코딩된 범주형 결합
X_train_encoded = pd.concat([
    X_train[numeric_cols].reset_index(drop=True),
    pd.DataFrame(X_train_cat, columns=encoded_cat_cols),
], axis=1)

X_test_encoded = pd.concat([
    X_test[numeric_cols].reset_index(drop=True),
    pd.DataFrame(X_test_cat, columns=encoded_cat_cols),
], axis=1)

# 기존 방식(원본 수치형 → 원본 범주 순서의 더미)과 같은 피처 순서를 유지한다.
# 열 순서 때문에 트리 계열의 무작위 열 샘플링 결과가 달라지는 것을 방지한다.
ordered_feature_cols = list(original_numeric_cols)
for col in string_cols:
    if col in binary_cols:
        ordered_feature_cols.append(col)
    else:
        ordered_feature_cols.extend([c for c in encoded_cat_cols if c.startswith(f'{col}_')])

X_train_encoded = X_train_encoded[ordered_feature_cols]
X_test_encoded = X_test_encoded[ordered_feature_cols]

# 3-1. 중복 더미 제거
# 부가서비스 6개의 'No internet service'는 InternetService='No'와 완전히 같은 집단이고,
# MultipleLines의 'No phone service'는 PhoneService='No'와 같다.
# 원핫인코딩하면 같은 정보가 여러 열로 반복되어 중요도·회귀계수 해석이 흐려진다.
redundant_keys = ['No internet service', 'No phone service']

# 제거 전에 실제로 중복인지 확인 (같은 키를 가진 더미끼리 상관 1.0)
for key in redundant_keys:
    dup_cols = [c for c in X_train_encoded.columns if key in c]
    if len(dup_cols) > 1:
        corr_min = X_train_encoded[dup_cols].corr().min().min()
        print(f"'{key}' 더미 {len(dup_cols)}개 최소 상관: {corr_min:.4f}")
    else:
        print(f"'{key}' 더미 {len(dup_cols)}개 (중복 없음)")

drop_dummy_cols = [c for c in X_train_encoded.columns if any(k in c for k in redundant_keys)]
print('\n제거할 중복 더미:', drop_dummy_cols)

X_train_encoded = X_train_encoded.drop(columns=drop_dummy_cols)
X_test_encoded = X_test_encoded.drop(columns=drop_dummy_cols)
print(f'피처 수: {len(drop_dummy_cols) + X_train_encoded.shape[1]} → {X_train_encoded.shape[1]}')

# 인터넷·전화 미사용 집단은 InternetService_No, PhoneService_Yes 로 여전히 구분된다.

# 표준화 (학습 데이터에만 fit, 테스트는 transform)
# 로지스틱 회귀는 L2 규제와 특성 스케일에 민감하므로 StandardScaler를 사용한다.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_encoded)
X_test_scaled = scaler.transform(X_test_encoded)



print('학습에 사용할 컬럼 수:', X_train_encoded.shape[1])
X_train_encoded.head()


# 4. 모델 비교 (기본 파라미터)
# 기존 로지스틱 회귀·랜덤포레스트에 XGBoost를 추가한다. (ml2 분류 예제 참고)
def evaluate_clf(name, clf, X_tr, y_tr, X_te, y_te):
    pred = clf.predict(X_te)
    prob = clf.predict_proba(X_te)[:, 1]
    result = {
        'model': name,
        'train_acc': accuracy_score(y_tr, clf.predict(X_tr)),
        'test_acc': accuracy_score(y_te, pred),
        'prec': precision_score(y_te, pred, zero_division=0),
        'rec': recall_score(y_te, pred, zero_division=0),
        'f1': f1_score(y_te, pred, zero_division=0),
        'auc': roc_auc_score(y_te, prob),
    }
    print(f"\n[{name}]")
    print('Train Accuracy:', round(result['train_acc'], 4))
    print('Test Accuracy :', round(result['test_acc'], 4))
    print('Precision(이탈):', round(result['prec'], 4))
    print('Recall(이탈)   :', round(result['rec'], 4))
    print('F1 (이탈)     :', round(result['f1'], 4))
    print('ROC-AUC       :', round(result['auc'], 4))
    print(classification_report(y_te, pred, target_names=['유지', '이탈']))
    return result


default_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(
        n_estimators=100,
        eval_metric='logloss',
        random_state=42,
    ),
}

default_rows = []
fitted_default = {}
for name, clf in default_models.items():
    clf.fit(X_train_scaled, y_train)
    fitted_default[name] = clf
    default_rows.append(evaluate_clf(name, clf, X_train_scaled, y_train, X_test_scaled, y_test))

print('\n=== 기본 파라미터 비교 ===')
print(pd.DataFrame(default_rows).to_string(index=False))


# 5. GridSearchCV 하이퍼파라미터 튜닝 (ml2와 같은 방식)
# cv=3, n_jobs=-1, 분류이므로 scoring='roc_auc'
search_space = {
    'Logistic Regression': (
        LogisticRegression(max_iter=1000, random_state=42),
        {
            'C': [0.01, 0.1, 1, 10],
            'class_weight': [None, 'balanced'],
        },
    ),
    'Random Forest': (
        RandomForestClassifier(random_state=42),
        {
            'n_estimators': [50, 100, 200],
            'max_depth': [None, 10, 20],
            'min_samples_split': [2, 5, 10],
        },
    ),
    'XGBoost': (
        XGBClassifier(eval_metric='logloss', random_state=42),
        {
            'n_estimators': [100, 200],
            'max_depth': [3, 5],
            'learning_rate': [0.05, 0.1],
            'subsample': [0.8, 1],
            'colsample_bytree': [0.8, 1],
        },
    ),
}

tuned_models = {}
tuned_rows = []
for name, (estimator, param_grid) in search_space.items():
    print(f'\n===== GridSearchCV: {name} =====')
    grid = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        cv=3,
        scoring='roc_auc',
        n_jobs=-1,
        verbose=1,
    )
    grid.fit(X_train_scaled, y_train)
    print('최적 하이퍼파라미터:', grid.best_params_)
    print('CV ROC-AUC:', round(grid.best_score_, 4))
    best_clf = grid.best_estimator_
    tuned_models[name] = best_clf
    row = evaluate_clf(f'{name} (tuned)', best_clf, X_train_scaled, y_train, X_test_scaled, y_test)
    row['best_params'] = grid.best_params_
    row['cv_auc'] = grid.best_score_
    tuned_rows.append(row)

compare_df = pd.DataFrame(tuned_rows)
print('\n=== 튜닝 후 모델 비교 ===')
print(compare_df.drop(columns=['best_params']).to_string(index=False))

# 5-1. 모델별 튜닝 전후 수치 비교
before_map = {row['model']: row for row in default_rows}
after_map = {row['model'].replace(' (tuned)', ''): row for row in tuned_rows}

before_after_rows = []
for name in default_models:
    b, a = before_map[name], after_map[name]
    before_after_rows.append({
        'model': name,
        'train_acc_before': b['train_acc'],
        'train_acc_after': a['train_acc'],
        'test_acc_before': b['test_acc'],
        'test_acc_after': a['test_acc'],
        'prec_before': b['prec'],
        'prec_after': a['prec'],
        'rec_before': b['rec'],
        'rec_after': a['rec'],
        'f1_before': b['f1'],
        'f1_after': a['f1'],
        'auc_before': b['auc'],
        'auc_after': a['auc'],
        'gap_before': b['train_acc'] - b['test_acc'],
        'gap_after': a['train_acc'] - a['test_acc'],
        'delta_auc': a['auc'] - b['auc'],
        'delta_gap': (a['train_acc'] - a['test_acc']) - (b['train_acc'] - b['test_acc']),
    })

ba_df = pd.DataFrame(before_after_rows)
print('\n=== 튜닝 전후 비교 (테스트 세트) ===')
print(
    ba_df[[
        'model',
        'test_acc_before', 'test_acc_after',
        'prec_before', 'prec_after',
        'rec_before', 'rec_after',
        'f1_before', 'f1_after',
        'auc_before', 'auc_after', 'delta_auc',
    ]].round(4).to_string(index=False)
)
print('\n=== 튜닝 전후 과적합 갭 (Train Acc - Test Acc) ===')
print(
    ba_df[[
        'model',
        'train_acc_before', 'train_acc_after',
        'test_acc_before', 'test_acc_after',
        'gap_before', 'gap_after', 'delta_gap',
    ]].round(4).to_string(index=False)
)

# 튜닝 전후 AUC · 과적합 갭 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
x = np.arange(len(ba_df))
w = 0.35
axes[0].bar(x - w / 2, ba_df['auc_before'], w, label='튜닝 전')
axes[0].bar(x + w / 2, ba_df['auc_after'], w, label='튜닝 후')
axes[0].set_xticks(x)
axes[0].set_xticklabels(ba_df['model'], rotation=15)
axes[0].set_ylim(0.80, 0.86)
axes[0].set_ylabel('ROC-AUC')
axes[0].set_title('모델별 튜닝 전후 테스트 ROC-AUC')
axes[0].legend()
axes[0].grid(True, axis='y', alpha=0.3)

axes[1].bar(x - w / 2, ba_df['gap_before'], w, label='튜닝 전')
axes[1].bar(x + w / 2, ba_df['gap_after'], w, label='튜닝 후')
axes[1].set_xticks(x)
axes[1].set_xticklabels(ba_df['model'], rotation=15)
axes[1].set_ylabel('Train Acc - Test Acc')
axes[1].set_title('모델별 튜닝 전후 과적합 갭')
axes[1].legend()
axes[1].grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


# 테스트 ROC-AUC가 가장 높은 모델을 최종 모델로 사용
best_name = compare_df.loc[compare_df['auc'].idxmax(), 'model'].replace(' (tuned)', '')
model = tuned_models[best_name]
print(f'\n최종 모델: {best_name}')
print('최종 하이퍼파라미터:', compare_df.loc[compare_df['auc'].idxmax(), 'best_params'])


# 5-2. 운영 임계값 선택
# 테스트 세트로 임계값을 고르면 테스트 정보가 학습에 섞이므로,
# 학습 데이터의 3-Fold OOF(out-of-fold) 확률만 사용한다.
# 실제 이탈을 덜 놓치기 위해 Recall >= 0.70인 후보 중 F1이 가장 높은 값을 선택한다.
threshold_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
oof_prob = cross_val_predict(
    model,
    X_train_scaled,
    y_train,
    cv=threshold_cv,
    method='predict_proba',
    n_jobs=-1,
)[:, 1]

threshold_rows = []
for threshold_i in np.arange(0.10, 0.91, 0.01):
    oof_pred_i = (oof_prob >= threshold_i).astype(int)
    threshold_rows.append({
        'threshold': threshold_i,
        'precision': precision_score(y_train, oof_pred_i, zero_division=0),
        'recall': recall_score(y_train, oof_pred_i, zero_division=0),
        'f1': f1_score(y_train, oof_pred_i, zero_division=0),
    })

threshold_df = pd.DataFrame(threshold_rows)
valid_thresholds = threshold_df[threshold_df['recall'] >= 0.70]
if valid_thresholds.empty:
    decision_threshold = 0.50
    print('Recall 0.70 이상 임계값을 찾지 못해 기본값 0.50을 사용합니다.')
else:
    threshold_best_idx = valid_thresholds.sort_values(
        ['f1', 'precision', 'threshold'], ascending=[False, False, False]
    ).index[0]
    decision_threshold = round(float(threshold_df.loc[threshold_best_idx, 'threshold']), 2)

selected_oof = threshold_df.iloc[(threshold_df['threshold'] - decision_threshold).abs().argsort()[:1]]
print(f'\n운영 임계값: {decision_threshold:.2f} (학습 OOF Recall>=0.70 중 F1 최대)')
print(selected_oof.round(4).to_string(index=False))

plt.figure(figsize=(8, 5))
plt.plot(threshold_df['threshold'], threshold_df['precision'], label='Precision')
plt.plot(threshold_df['threshold'], threshold_df['recall'], label='Recall')
plt.plot(threshold_df['threshold'], threshold_df['f1'], label='F1')
plt.axvline(decision_threshold, color='red', linestyle='--', label=f'선택 임계값={decision_threshold:.2f}')
plt.xlabel('Decision Threshold')
plt.ylabel('Score')
plt.title('학습 OOF 기준 임계값별 Precision / Recall / F1')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


# 5-3. 최종 모델 평가: 기본 임계값 0.50과 운영 임계값을 함께 비교
y_prob = model.predict_proba(X_test_scaled)[:, 1]
y_pred_default = (y_prob >= 0.50).astype(int)
y_pred = (y_prob >= decision_threshold).astype(int)
train_accuracy = accuracy_score(y_train, model.predict(X_train_scaled))

def threshold_metrics(label, pred):
    cm_i = confusion_matrix(y_test, pred)
    tn_i, fp_i, fn_i, tp_i = cm_i.ravel()
    result = {
        '기준': label,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred, zero_division=0),
        'Recall': recall_score(y_test, pred, zero_division=0),
        'F1': f1_score(y_test, pred, zero_division=0),
        'TN': tn_i,
        'FP': fp_i,
        'FN': fn_i,
        'TP': tp_i,
    }
    return result, cm_i

result_default, cm_default = threshold_metrics('기본 0.50', y_pred_default)
result_operation, cm = threshold_metrics(f'운영 {decision_threshold:.2f}', y_pred)
threshold_compare_df = pd.DataFrame([result_default, result_operation])

print(f'\nTrain Accuracy: {train_accuracy:.4f}')
print('\n=== 기본/운영 임계값 테스트 성능 비교 ===')
print(threshold_compare_df.round(4).to_string(index=False))
print('\n운영 임계값 Classification Report:')
print(classification_report(y_test, y_pred, target_names=['유지', '이탈']))

# 기본값과 운영값 혼동 행렬 비교
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
for ax, cm_i, title in [
    (axes[0], cm_default, '기본 임계값 0.50'),
    (axes[1], cm, f'운영 임계값 {decision_threshold:.2f}'),
]:
    sns.heatmap(
        cm_i,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=['유지', '이탈'],
        yticklabels=['유지', '이탈'],
        cbar=False,
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel('예측값')
    ax.set_ylabel('실제값')
plt.suptitle(f'혼동 행렬 비교 ({best_name})')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'운영 기준 TN {tn} / FP {fp} / FN {fn} / TP {tp}')
print(f'특이도(유지를 유지로): {tn / (tn + fp):.4f}')
print(f'재현율(이탈을 이탈로): {tp / (tp + fn):.4f}')


# 6. 중요 변수 확인
# 로지스틱 회귀는 표준화 회귀계수, 트리 계열은 Feature Importance
if hasattr(model, 'coef_'):
    coef_df = pd.DataFrame({
        'Feature': X_train_encoded.columns,
        'Coefficient': model.coef_[0],
    })
    coef_df['AbsCoefficient'] = coef_df['Coefficient'].abs()
    coef_df = coef_df.sort_values(by='AbsCoefficient', ascending=False)
    print(coef_df.head(10))

    plt.figure(figsize=(10, 5))
    sns.barplot(x='Coefficient', y='Feature', data=coef_df.head(10))
    plt.axvline(0, color='gray', linewidth=0.8)
    plt.title('상위 10개 표준화 회귀계수 (절댓값 기준)')
    plt.show()
else:
    imp_df = pd.DataFrame({
        'Feature': X_train_encoded.columns,
        'Importance': model.feature_importances_,
    }).sort_values(by='Importance', ascending=False)
    print(imp_df.head(10))

    plt.figure(figsize=(10, 5))
    sns.barplot(x='Importance', y='Feature', data=imp_df.head(10))
    plt.title(f'상위 10개 특성 중요도 ({best_name})')
    plt.show()


# 7. ROC 커브 시각화 (튜닝된 3개 모델 비교)
plt.figure(figsize=(8, 6))
for name, clf in tuned_models.items():
    y_prob_i = clf.predict_proba(X_test_scaled)[:, 1]
    fpr_i, tpr_i, _ = roc_curve(y_test, y_prob_i)
    auc_i = auc(fpr_i, tpr_i)
    plt.plot(fpr_i, tpr_i, lw=2, label=f'{name} (AUC = {auc_i:.2f})')

plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('ROC Curve (GridSearchCV 튜닝 후)')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()




# 새로운 고객 1명으로 예측 연습 (운영 임계값 적용)
sample = X_test_encoded.iloc[0:1]
sample_scaled = scaler.transform(sample)
prediction_prob = model.predict_proba(sample_scaled)
prediction = int(prediction_prob[0][1] >= decision_threshold)

print('예측 결과:', '이탈' if prediction == 1 else '유지')
print(f'이탈 확률: {prediction_prob[0][1]:.2%}')
print(f'적용 임계값: {decision_threshold:.2f}')
print('실제값:', '이탈' if y_test.iloc[0] == 1 else '유지')




# 8. 모델과 전처리 객체 저장
os.makedirs('model', exist_ok=True)
joblib.dump(encoder, 'model/telco_encoder.joblib')
joblib.dump(scaler, 'model/telco_scaler.joblib')
joblib.dump(model, 'model/telco_model.joblib')
joblib.dump(
    {
        'numeric_cols': numeric_cols,
        'categorical_cols': categorical_cols,
        'binary_mappings': binary_mappings,
        'feature_names': list(X_train_encoded.columns),
        'model_name': best_name,
        'decision_threshold': decision_threshold,
        'threshold_rule': '3-Fold OOF Recall >= 0.70 중 F1 최대',
    },
    'model/telco_features_meta.joblib',
)

print("모델과 전처리 객체가 'model/' 폴더에 성공적으로 저장되었습니다.")





데이터 크기: (7043, 21)
<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   

C:\Users\human-06\AppData\Local\Temp\ipykernel_12732\16074434.py:91: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\human-06\AppData\Local\Temp\ipykernel_12732\16074434.py:103: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\human-06\AppData\Local\Temp\ipykernel_12732\16074434.py:110: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\human-06\AppData\Local\Temp\ipykernel_12732\16074434.py:117: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


자동 이진화 대상 및 매핑:
  gender: {'Female': 0, 'Male': 1}
  Partner: {'No': 0, 'Yes': 1}
  Dependents: {'No': 0, 'Yes': 1}
  PhoneService: {'No': 0, 'Yes': 1}
  PaperlessBilling: {'No': 0, 'Yes': 1}
기존 이진 수치형: SeniorCitizen {'해당 없음': 0, '고령': 1}
타깃 이진화: Churn {'No': 0, 'Yes': 1}
수치형(이진화 열 포함): ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'PaperlessBilling', 'MonthlyCharges', 'TotalCharges']
원핫인코딩 대상: ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod']

학습 데이터: (5634, 19)
테스트 데이터: (1409, 19)
'No internet service' 더미 6개 최소 상관: 1.0000
'No phone service' 더미 1개 (중복 없음)

제거할 중복 더미: ['MultipleLines_No phone service', 'OnlineSecurity_No internet service', 'OnlineBackup_No internet service', 'DeviceProtection_No internet service', 'TechSupport_No internet service', 'StreamingTV_No internet service', 'StreamingMovies_No internet service']
피처 수: 30 → 23
학


[Random Forest]
Train Accuracy: 0.998
Test Accuracy : 0.7935
Precision(이탈): 0.6436
Recall(이탈)   : 0.4973
F1 (이탈)     : 0.5611
ROC-AUC       : 0.8287
              precision    recall  f1-score   support

          유지       0.83      0.90      0.86      1035
          이탈       0.64      0.50      0.56       374

    accuracy                           0.79      1409
   macro avg       0.74      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409



...


최종 모델: XGBoost
최종 하이퍼파라미터: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 200, 'subsample': 0.8}



운영 임계값: 0.33 (학습 OOF Recall>=0.70 중 F1 최대)
 threshold  precision  recall   f1
      0.33       0.56    0.74 0.64

Train Accuracy: 0.8223

=== 기본/운영 임계값 테스트 성능 비교 ===
     기준  Accuracy  Precision  Recall   F1  TN  FP  FN  TP
기본 0.50      0.80       0.65    0.53 0.58 931 104 177 197
운영 0.33      0.76       0.54    0.74 0.63 799 236  96 278

운영 임계값 Classification Report:
              precision    recall  f1-score   support

          유지       0.89      0.77      0.83      1035
          이탈       0.54      0.74      0.63       374

    accuracy                           0.76      1409
   macro avg       0.72      0.76      0.73      1409
weighted avg       0.80      0.76      0.77      1409



C:\Users\human-06\AppData\Local\Temp\ipykernel_12732\16074434.py:472: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


운영 기준 TN 799 / FP 236 / FN 96 / TP 278
특이도(유지를 유지로): 0.7720
재현율(이탈을 이탈로): 0.7433
                           Feature  Importance
18               Contract_Two year        0.18
9      InternetService_Fiber optic        0.16
21  PaymentMethod_Electronic check        0.13
17               Contract_One year        0.12
10              InternetService_No        0.07
1                           tenure        0.07
19                PaperlessBilling        0.03
16             StreamingMovies_Yes        0.03
15                 StreamingTV_Yes        0.02
11              OnlineSecurity_Yes        0.02
예측 결과: 유지
이탈 확률: 1.39%
적용 임계값: 0.33
실제값: 유지
모델과 전처리 객체가 'model/' 폴더에 성공적으로 저장되었습니다.


C:\Users\human-06\AppData\Local\Temp\ipykernel_12732\16074434.py:528: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\human-06\AppData\Local\Temp\ipykernel_12732\16074434.py:562: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\human-06\AppData\Local\Temp\ipykernel_12732\16074434.py:581: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
